# Task 6: NER on WNUT-17 Emerging Entities Dataset

This notebook repeats the Task 1–5 pipeline (EDA ->feature engineering -> CRF training -> error analysis -> business impact) on the **WNUT-17** dataset, which focuses on *emerging, rare, and noisy entities* from user-generated text (tweets, Reddit, YouTube comments)- a closer proxy for real customer support tickets than newswire text.

**WNUT-17 entity types:** `person`, `location`, `corporation`, `product`, `creative-work`, `group`.

In [1]:
!pip install datasets scikit-learn sklearn-crfsuite matplotlib -q


## 6.0 Setup

Install/import dependencies and load the dataset.

In [2]:
import matplotlib.pyplot as plt
from collections import Counter
from datasets import DownloadConfig, load_dataset
import sklearn_crfsuite
from sklearn_crfsuite import metrics

wnut = load_dataset("tner/wnut2017")
label_names = wnut["train"].features["ner_tags"].feature.names
print(label_names)
print(wnut)

RuntimeError: Dataset scripts are no longer supported, but found wnut2017.py

## 6.1 Text Preprocessing & EDA

Same statistics as Task 1: sentence counts, vocabulary size, sentence-length distribution, and entity-tag frequency.

In [ ]:
train_sentences = wnut["train"]["tokens"]
val_sentences = wnut["validation"]["tokens"]
test_sentences = wnut["test"]["tokens"]

train_lengths = [len(s) for s in train_sentences]
all_train_tokens = [tok for sent in train_sentences for tok in sent]
vocab = set(all_train_tokens)

print("--- WNUT-17 Dataset Statistics ---")
print(f"Number of training sentences:   {len(train_sentences)}")
print(f"Number of validation sentences: {len(val_sentences)}")
print(f"Number of test sentences:       {len(test_sentences)}")
print(f"Vocabulary size (train):        {len(vocab)}")
print(f"Avg sentence length:            {sum(train_lengths)/len(train_lengths):.2f}")
print(f"Max sentence length:            {max(train_lengths)}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(train_lengths, bins=30, color='steelblue', edgecolor='black')
plt.title("Sentence Length Distribution - WNUT-17 (Train)")
plt.xlabel("Tokens per sentence")
plt.ylabel("Count")
plt.show()

In [ ]:
all_train_tags = [label_names[tag] for sent in wnut["train"]["ner_tags"] for tag in sent]
tag_counts = Counter(all_train_tags)
tag_counts_no_o = {k: v for k, v in tag_counts.items() if k != "O"}

plt.figure(figsize=(10, 5))
plt.bar(tag_counts_no_o.keys(), tag_counts_no_o.values(), color='darkorange', edgecolor='black')
plt.title("Entity Tag Frequency - WNUT-17 (excluding 'O')")
plt.xticks(rotation=45)
plt.ylabel("Frequency")
plt.show()

print(f"'O' tag proportion: {tag_counts['O'] / len(all_train_tags):.2%}")

**Interpretation:** WNUT-17 is far more imbalanced and sparse than CoNLL-2003. Entity classes are rarer, and the vocabulary is noisier (slang, hashtags, misspellings typical of social media). This mirrors real support-ticket text more closely and is expected to be harder for a feature-based CRF.

In [ ]:
word_counts = Counter(all_train_tokens)
top30 = word_counts.most_common(30)
words, counts = zip(*top30)

plt.figure(figsize=(12, 5))
plt.bar(words, counts, color='seagreen', edgecolor='black')
plt.title("Top 30 Most Frequent Words - WNUT-17 (Train)")
plt.xticks(rotation=60)
plt.show()

least30 = word_counts.most_common()[:-31:-1]
lwords, lcounts = zip(*least30)

plt.figure(figsize=(12, 5))
plt.bar(lwords, lcounts, color='indianred', edgecolor='black')
plt.title("Bottom 30 Least Frequent Words - WNUT-17 (Train)")
plt.xticks(rotation=60)
plt.show()

**Interpretation:** As with CoNLL-2003, most rare/least-frequent words are hapax legomena - usernames, hashtags, and one-off slang terms. These are exactly the "emerging entity" tokens WNUT-17 is designed to test, and they are the hardest for any model relying on memorized surface forms.

## 6.2 Feature Engineering

Same feature set as Task 2 (orthographic, affix, and context-window features), reused here unchanged so results are comparable to the CoNLL-2003 CRF.

- **Lowercase / word identity** - normalizes casing noise.
- **Capitalization (`istitle`, `isupper`)** - proper-noun signal, still useful even in noisy text.
- **`isdigit`** - flags numeric tokens (prices, dates in tickets).
- **Prefix/suffix (last 2–3 chars)** - captures morphology and helps with unseen/misspelled words.
- **Previous/next word context** - critical here since WNUT-17 entities are often only identifiable from surrounding words (e.g., a novel product name next to "just launched").
- **BOS/EOS** - sentence boundary flags.

In [ ]:
def word2features(sent, i):
    word = sent[i]
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word[-2:]': word[-2:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
    }
    if i > 0:
        prev_word = sent[i-1]
        features.update({
            '-1:word.lower()': prev_word.lower(),
            '-1:word.istitle()': prev_word.istitle(),
            '-1:word.isupper()': prev_word.isupper(),
        })
    else:
        features['BOS'] = True

    if i < len(sent) - 1:
        next_word = sent[i+1]
        features.update({
            '+1:word.lower()': next_word.lower(),
            '+1:word.istitle()': next_word.istitle(),
            '+1:word.isupper()': next_word.isupper(),
        })
    else:
        features['EOS'] = True
    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(tags):
    return [label_names[t] for t in tags]

In [ ]:
X_train = [sent2features(s) for s in wnut["train"]["tokens"]]
y_train = [sent2labels(t) for t in wnut["train"]["ner_tags"]]

X_val = [sent2features(s) for s in wnut["validation"]["tokens"]]
y_val = [sent2labels(t) for t in wnut["validation"]["ner_tags"]]

X_test = [sent2features(s) for s in wnut["test"]["tokens"]]
y_test = [sent2labels(t) for t in wnut["test"]["ner_tags"]]

print(X_train[0][0])

## 6.3 Named Entity Recognition - CRF Model

Same CRF configuration as the CoNLL-2003 model (`lbfgs`, L1/L2 regularization, all possible transitions) for a fair comparison.

In [ ]:
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)
crf.fit(X_train, y_train)

In [ ]:
y_pred = crf.predict(X_test)

labels = list(crf.classes_)
labels.remove('O')

report = metrics.flat_classification_report(y_test, y_pred, labels=labels, digits=3)
print(report)

In [ ]:
f1_scores = metrics.flat_f1_score(y_test, y_pred, labels=labels, average=None)

plt.figure(figsize=(12, 5))
plt.bar(labels, f1_scores, color='mediumpurple', edgecolor='black')
plt.title("Per-Entity F1 Score - WNUT-17 CRF")
plt.xticks(rotation=45)
plt.ylabel("F1 Score")
plt.ylim(0, 1)
plt.show()

**Discussion - hardest entity types:** `corporation`, `product`, and `creative-work` are typically the weakest classes on WNUT-17, since they are the most "emerging"/open-vocabulary categories (new brand names, songs, apps) with the fewest training examples and the least consistent orthographic signal. `person` and `location` tend to perform best, since capitalization and common name patterns still apply. Overall F1 on WNUT-17 is expected to be noticeably lower than on CoNLL-2003.

## 6.4 Error Analysis

Sample predicted vs. true labels on the test set, and a summary of the most frequent error patterns.

In [ ]:
def print_sample_errors(n_sentences=3):
    shown = 0
    for tokens, true_tags, pred_tags in zip(wnut["test"]["tokens"], y_test, y_pred):
        if true_tags != pred_tags:
            print("Tokens: ", tokens)
            print("True:   ", true_tags)
            print("Pred:   ", pred_tags)
            print("-" * 60)
            shown += 1
        if shown >= n_sentences:
            break

print_sample_errors()

### Top 3 Error Patterns Observed (WNUT-17 CRF)

1. **Missed emerging entities (false negatives on rare classes):** Novel `product`/`corporation`/`creative-work` names not seen in training are tagged `O` almost by default - the model has no surface-form memory to fall back on and lacks deep semantic understanding.
2. **Boundary errors on multi-word entities:** Similar to CoNLL-2003, `B-` tags are often correct but trailing `I-` tokens of a multi-word entity (e.g., a multi-word product or group name) get dropped, truncating the span.
3. **Confusion between `group`/`corporation`/`person`:** Informal text makes these three ambiguous (e.g., a band name that could read as a person or a group), and hashtags/@mentions add noise that the CRF's shallow orthographic features can't resolve.

## 6.5 Business Impact - Emerging Entities in Customer Support

### Recommendation (WNUT-17-informed)

Deploying an NER model trained on emerging-entity data like WNUT-17 is directly relevant to customer support, where tickets constantly reference brand-new products, informal company nicknames, and slang that a newswire-trained model (like the CoNLL-2003 CRF) would miss entirely. The WNUT-17 results show that while `person` and `location` extraction remains reliable, performance on `product` and `corporation` entities is weak - precisely the categories most useful for routing tickets to the right product team or account owner. We recommend treating this model as a **secondary "novelty detector"** layered on top of the CoNLL-based system: flagging tokens the primary model misses so they can be reviewed and added to a growing internal entity dictionary. Over time, this human-in-the-loop process builds a domain-specific gazetteer of product names and company aliases, closing the gap that generic training data can't cover. Long-term, migrating to a transformer-based model fine-tuned on actual ticket text - rather than relying on hand-crafted orthographic features - is the clearest path to reliably catching new entities without constant manual upkeep.

### Limitations
- Severe class imbalance and few training examples for `product`/`corporation`/`creative-work`.
- Informal/noisy text (typos, hashtags, abbreviations) breaks orthographic features like capitalization.
- No inherent mechanism to generalize to genuinely unseen entity names beyond context cues.

### Future Improvements
- Fine-tune a transformer (e.g., BERTweet, RoBERTa) pretrained on social/informal text.
- Maintain an evolving gazetteer of product/company names sourced from confirmed tickets.
- Add character-level embeddings to better handle misspellings and novel tokens.

## 6.6 Summary: CoNLL-2003 vs. WNUT-17

| Aspect | CoNLL-2003 | WNUT-17 |
|---|---|---|
| Text style | Clean newswire | Noisy, informal social text |
| Entity types | PER, ORG, LOC, MISC | person, location, corporation, product, creative-work, group |
| Class imbalance | High | Very high |
| CRF performance | Strong, especially PER/LOC | Weaker overall, especially on product/corporation/creative-work |
| Relevance to support tickets | Moderate (formal language) | High (informal, novel entities) |

**Takeaway:** WNUT-17 is the more realistic proxy for customer support text. Its lower CRF performance highlights exactly why a production system should combine a CRF/BiLSTM baseline with mechanisms for catching novel entities - either a growing gazetteer or a transformer fine-tuned on real ticket data.